# shift_right原则

input右移一位对齐label
输入：input_ids[:,:-1] --（batch_size,seq_len）
标签：input_ids[:,1:]

In [1]:
import torch 
import torch.nn as nn

def sft_loss(logits,labels,ignore_index=-100):
    """
    计算SFT的交叉熵损失 自动处理shift right 并且忽略padding 
    参数：
        logits: 模型输出的 logits,形状(batch_size,seq_len,vocab_size)
        labels: 真实token_ids, 形状(batch_size,seq_len),其中padding部分的token_id为ignore_index(padding是为了让同一个batch的各个序列的长度一致)
    返回：
        标量损失
    """
    shift_logits = logits[..., :-1, :].contiguous()  # 去掉最后一个时间步的logits
    shift_labels = labels[..., 1:].contiguous()  # 去掉第一个时间步的labels

    loss_function = nn.CrossEntropyLoss(ignore_index=ignore_index)
    loss = loss_function(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)) #把batch和seq_len维度合并
    return loss

if __name__ == "__main__":
    batch_size, seq_len, vocab_size = 2, 5, 10
    logits = torch.randn(batch_size, seq_len, vocab_size)  # 模拟模型输出
    labels = torch.randint(0, vocab_size, (batch_size, seq_len))
    labels[0, 3:] = -100  # 模拟padding部分
    loss = sft_loss(logits, labels)
    print("Loss:", loss.item())



Loss: 2.425140380859375
